In [1]:
import pandas as pd
import torch
from tqdm import tqdm
import torch.nn as nn
from torchvision import transforms
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import os
import numpy as np
from sklearn.metrics import f1_score, cohen_kappa_score, recall_score, accuracy_score, classification_report
from scipy.stats import spearmanr
from torch.utils.data import WeightedRandomSampler


# The following code will only execute
# successfully when compression is complete

import kagglehub

# Download latest version
path = kagglehub.dataset_download("mohabenloughmari/miccai-task2-full")

print("Path to dataset files:", path)

import warnings
warnings.filterwarnings("ignore")

base_path = path
path = os.path.join(base_path, 'Task_2')

df_train = pd.read_csv(os.path.join(path, "df_task2_train.csv"))
df_val = pd.read_csv(os.path.join(path, "df_task2_val.csv"))
df_test = pd.read_csv(os.path.join(path, "df_task2_test.csv"))

device = torch.device("cuda" if torch.cuda.is_available() else 'cpu')


Using Colab cache for faster access to the 'miccai-task2-full' dataset.
Path to dataset files: /kaggle/input/miccai-task2-full


In [2]:

labels = df_train['label'].values.astype(int)
class_counts = np.bincount(labels)
sample_weights = 1.0 / class_counts[labels]
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))

n_labels = df_train['label'].nunique()

tabular_cols = ['case', 'label', 'LOCALIZER', 'split_type', 'image']
tab_train = df_train.drop(columns=tabular_cols)
tab_val = df_val.drop(columns=tabular_cols)
tab_test = df_test.drop(columns=tabular_cols)

cat_cols = tab_train.select_dtypes(include='object').columns.tolist()
num_cols = tab_train.select_dtypes(exclude='object').columns.tolist()

print(f"Categorical columns: {cat_cols}")
print(f"Numerical columns: {num_cols}")

tab_train_encoded = pd.get_dummies(tab_train, columns=cat_cols, dtype=np.float32)
tab_val_encoded = pd.get_dummies(tab_val, columns=cat_cols, dtype=np.float32)
tab_test_encoded = pd.get_dummies(tab_test, columns=cat_cols, dtype=np.float32)

tab_val_encoded = tab_val_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)
tab_test_encoded = tab_test_encoded.reindex(columns=tab_train_encoded.columns, fill_value=0)



Categorical columns: ['side_eye', 'sex']
Numerical columns: ['BScan', 'age', 'num_current_visit']


In [3]:

train_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=10),
    transforms.ColorJitter(contrast=0.3),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])


In [4]:

class CustomImageDataset(Dataset):
    def __init__(self, dataframe, root_dir, tab_data, transform=None):
        self.dataframe = dataframe
        self.root_dir = root_dir
        self.tab_data = torch.tensor(tab_data.values, dtype=torch.float32)
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        img_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['image'])
        localiser_name = os.path.join(self.root_dir, self.dataframe.iloc[idx]['LOCALIZER'])

        image = Image.open(img_name).convert('RGB')
        localiser = Image.open(localiser_name).convert('RGB')

        if self.transform:
            image = self.transform(image)
            localiser = self.transform(localiser)

        label = self.dataframe.iloc[idx]['label']
        tab = self.tab_data[idx]
        return image, localiser, tab, label

In [5]:
train_ds = CustomImageDataset(df_train, os.path.join(path, 'train'), tab_train_encoded, transform=train_transform)  
val_ds = CustomImageDataset(df_val, os.path.join(path, 'val'), tab_val_encoded, transform=test_transform)  
test_ds = CustomImageDataset(df_test, os.path.join(path, 'test'), tab_test_encoded, transform=test_transform)  
  
labels = df_train['label'].values.astype(int)  
class_counts = np.bincount(labels)  
sample_weights = 1.0 / class_counts[labels]  
sampler = WeightedRandomSampler(sample_weights, len(sample_weights))  
  
train_loader = DataLoader(train_ds, batch_size=12, sampler=sampler)  
val_loader = DataLoader(val_ds, batch_size=12, shuffle=False)  
test_loader = DataLoader(test_ds, batch_size=12, shuffle=False)

In [6]:
from torchvision import models
#
#class ImageEncoder(nn.Module):
#    def __init__(self, embed_dim=768):
#        super().__init__()
#        weights = models.ConvNeXt_Base_Weights.DEFAULT  # Fixed: Large -> Base
#        self.backbone = models.convnext_base(weights=weights)
#        in_features = self.backbone.classifier[2].in_features  # 1024 for convnext_base
#        self.backbone.classifier = nn.Identity()
#        self.fc = nn.Linear(in_features, embed_dim)
#        for p in self.backbone.parameters():
#            p.requires_grad = True
#
#    def forward(self, x):
#        features = self.backbone(x)
#        return self.fc(features)
#

#class LocaliserEncoder(nn.Module):
#    def __init__(self, embed_dim=768):
#        super().__init__()
#        weights = models.ConvNeXt_Base_Weights.DEFAULT  # Fixed: Large -> Base
#        self.backbone = models.convnext_base(weights=weights)
#        in_features = self.backbone.classifier[2].in_features  # 1024 for convnext_base
#        self.backbone.classifier = nn.Identity()
#        self.fc = nn.Linear(in_features, embed_dim)
#        for p in self.backbone.features[-2:].parameters():
#            p.requires_grad = True
#
#    def forward(self, x):
#        features = self.backbone(x)
#        return self.fc(features)


class ImageEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.ConvNeXt_Base_Weights.DEFAULT
        self.backbone = models.convnext_base(weights=weights)
        in_features = self.backbone.classifier[2].in_features  # 1024
        self.backbone.classifier = nn.Sequential(
            nn.Flatten(1),  # flattens (batch, 1024, 1, 1) -> (batch, 1024)
        )
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class LocaliserEncoder(nn.Module):
    def __init__(self, embed_dim=768):
        super().__init__()
        weights = models.ConvNeXt_Base_Weights.DEFAULT
        self.backbone = models.convnext_base(weights=weights)
        in_features = self.backbone.classifier[2].in_features  # 1024
        self.backbone.classifier = nn.Sequential(
            nn.Flatten(1),  # flattens (batch, 1024, 1, 1) -> (batch, 1024)
        )
        self.fc = nn.Linear(in_features, embed_dim)
        for p in self.backbone.features[-2:].parameters():
            p.requires_grad = True

    def forward(self, x):
        features = self.backbone(x)
        return self.fc(features)


class TabularMLP(nn.Module):
    def __init__(self, tab_dim, d_model, dropout=0.3):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(tab_dim, 256),
            nn.BatchNorm1d(256),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(256, d_model),
            nn.BatchNorm1d(d_model),
            nn.GELU(),
            nn.Dropout(dropout),
        )

    def forward(self, tab):
        return self.mlp(tab)


class OCT_Pred(nn.Module):
    def __init__(self, n_labels, tab_dim, embed_dim=768, d_model=None, dropout=0.3):
        super().__init__()
        if d_model is None:
            d_model = embed_dim
        self.image_encoder = ImageEncoder(embed_dim)
        self.localiser_encoder = LocaliserEncoder(embed_dim)
        self.tab_encoder = TabularMLP(tab_dim, d_model, dropout)
        self.img_norm = nn.LayerNorm(embed_dim)
        self.loc_norm = nn.LayerNorm(embed_dim)
        fusion_dim = 2 * embed_dim + d_model
        self.mlp = nn.Sequential(
            nn.Linear(fusion_dim, 1024),
            nn.BatchNorm1d(1024),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(1024, 512),
            nn.BatchNorm1d(512),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(512, n_labels),
        )

    def forward(self, image, localiser, tab):
        img_features = self.image_encoder(image)
        loc_features = self.localiser_encoder(localiser)
        tab_features = self.tab_encoder(tab)
        combined = torch.cat([self.img_norm(img_features), self.loc_norm(loc_features), tab_features], dim=-1)
        output = self.mlp(combined)
        return output, combined

class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)

In [7]:
from sklearn.metrics import classification_report
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    all_labels = []
    all_predicted = []

    pbar = tqdm(loader, desc="Training", leave=False)
    for images, localisers, tab, labels in pbar:
        images = images.to(device)
        localisers = localisers.to(device)
        tab = tab.to(device)
        labels = labels.long().to(device)

        optimizer.zero_grad()
        outputs, _ = model(images, localisers, tab)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        _, predicted = outputs.max(1)
        correct += predicted.eq(labels).sum().item()
        total += labels.size(0)

        all_labels.extend(labels.cpu().numpy())
        all_predicted.extend(predicted.cpu().numpy())

    accuracy = 100. * correct / total
    print(classification_report(all_labels, all_predicted))
    return running_loss / len(loader), accuracy

In [8]:
from sklearn.metrics import f1_score, matthews_corrcoef, confusion_matrix, cohen_kappa_score, classification_report
import numpy as np

def specificity(class_ground_truth, class_prediction):
    eps = 0.000001
    cnf_matrix = confusion_matrix(class_ground_truth, class_prediction)
    FP = cnf_matrix.sum(axis=0) - np.diag(cnf_matrix)
    FN = cnf_matrix.sum(axis=1) - np.diag(cnf_matrix)
    TP = np.diag(cnf_matrix)
    TN = cnf_matrix.sum() - (FP + FN + TP)
    FP, FN, TP, TN = FP.astype(float), FN.astype(float), TP.astype(float), TN.astype(float)
    spe = TN / (TN + FP + eps)
    return spe.mean()

def evaluate(model, dataloader, criterion, device, show_report=True):
    model.eval()
    total_loss = 0
    correct = 0
    total = 0
    all_preds = []
    all_labels = []

    with torch.no_grad():
        pbar = tqdm(dataloader, desc='Evaluation', leave=False)
        for images, localisers, tab, labels in pbar:
            images = images.to(device)
            localisers = localisers.to(device)
            tab = tab.to(device)
            labels = labels.long().to(device)

            outputs, _ = model(images, localisers, tab)
            loss = criterion(outputs, labels)
            total_loss += loss.item()
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    accuracy = 100. * correct / total
    f1 = f1_score(all_labels, all_preds, average='micro')
    rk_corr = matthews_corrcoef(all_labels, all_preds)
    spec = specificity(all_labels, all_preds)
    qwk = cohen_kappa_score(all_labels, all_preds, weights='quadratic')
    score = 0.1 * f1 + 0.1 * spec + 0.6 * qwk + 0.2 * rk_corr

    if show_report:
        print("\n" + "="*80)
        print("CLASSIFICATION REPORT - Per-Class Performance:")
        print("="*80)
        print(classification_report(all_labels, all_preds,
                                    target_names=['Class 0', 'Class 1', 'Class 2'],
                                    digits=4))
        print("="*80 + "\n")

    return {
        'loss': total_loss / len(dataloader),
        'accuracy': accuracy,
        'F1_score': f1,
        'Rk-correlation': rk_corr,
        'Specificity': spec,
        'Quadratic-weighted_Kappa': qwk,
        'score': score
    }




In [9]:
#class FocalLoss(nn.Module):
#import torch.nn.functional as F
#    def __init__(self, gamma=2):
#        super().__init__()
#        self.gamma = gamma
#        
#    def forward(self, inputs, targets):
#        ce_loss = F.cross_entropy(inputs, targets, reduction='none')
#        pt = torch.exp(-ce_loss)
#        focal_loss = (1 - pt) ** self.gamma * ce_loss
#        return focal_loss.mean()
#criterion = FocalLoss(gamma=2)


In [10]:
model = OCT_Pred(n_labels=n_labels, tab_dim=7).to(device)
class_weights = torch.tensor(1.0 / class_counts, dtype=torch.float32).to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=5)
num_epochs = 5

best_score = -float('inf')

for epoch in range(num_epochs):
    print(f"\n{'='*50}")
    print(f"Epoch {epoch+1}/{num_epochs}")
    print(f"{'='*50}")

    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    val_metrics = evaluate(model, val_loader, criterion, device)
    scheduler.step()

    print(f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}%")
    print(f"Val Loss: {val_metrics['loss']:.4f} | Val Acc: {val_metrics['accuracy']:.2f}%")
    print(f"Val Score: {val_metrics['score']:.4f}")

    if val_metrics['score'] > best_score:
        best_score = val_metrics['score']
        best_model=model
        print(f"New best model saved! Score: {best_score:.4f}")



Epoch 1/5


              precision    recall  f1-score   support

           0       0.72      0.89      0.80      2693
           1       0.90      0.45      0.60      2753
           2       0.76      0.97      0.85      2636

    accuracy                           0.77      8082
   macro avg       0.79      0.77      0.75      8082
weighted avg       0.79      0.77      0.75      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.1267    0.2707    0.1726       351
     Class 1     0.7439    0.7682    0.7558      2821
     Class 2     0.1195    0.0292    0.0470       650

    accuracy                         0.5968      3822
   macro avg     0.3300    0.3560    0.3251      3822
weighted avg     0.5810    0.5968    0.5817      3822


Train Loss: 0.2399 | Train Acc: 76.68%
Val Loss: 1.3082 | Val Acc: 59.68%
Val Score: 0.1432
New best model saved! Score: 0.1432

Epoch 2/5


              precision    recall  f1-score   support

           0       0.86      0.98      0.92      2809
           1       0.98      0.74      0.84      2633
           2       0.91      0.99      0.95      2640

    accuracy                           0.91      8082
   macro avg       0.92      0.91      0.90      8082
weighted avg       0.91      0.91      0.90      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.1202    0.1795    0.1440       351
     Class 1     0.7467    0.6062    0.6691      2821
     Class 2     0.1379    0.2138    0.1677       650

    accuracy                         0.5003      3822
   macro avg     0.3350    0.3332    0.3269      3822
weighted avg     0.5856    0.5003    0.5356      3822


Train Loss: 0.0820 | Train Acc: 90.68%
Val Loss: 1.6392 | Val Acc: 50.03%
Val Score: 0.0906

Epoch 3/5


...


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.2902    0.1595    0.2059       351
     Class 1     0.7386    0.9046    0.8133      2821
     Class 2     0.0000    0.0000    0.0000       650

    accuracy                         0.6824      3822
   macro avg     0.3429    0.3547    0.3397      3822
weighted avg     0.5718    0.6824    0.6192      3822


Train Loss: 0.0383 | Train Acc: 95.87%
Val Loss: 1.3963 | Val Acc: 68.24%
Val Score: 0.1506
New best model saved! Score: 0.1506

Epoch 4/5


              precision    recall  f1-score   support

           0       0.97      1.00      0.98      2714
           1       1.00      0.95      0.97      2704
           2       0.99      1.00      0.99      2664

    accuracy                           0.98      8082
   macro avg       0.98      0.98      0.98      8082
weighted avg       0.98      0.98      0.98      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.3333    0.0541    0.0931       351
     Class 1     0.7401    0.9709    0.8399      2821
     Class 2     0.2969    0.0292    0.0532       650

    accuracy                         0.7266      3822
   macro avg     0.4568    0.3514    0.3288      3822
weighted avg     0.6273    0.7266    0.6376      3822


Train Loss: 0.0166 | Train Acc: 98.32%
Val Loss: 1.9806 | Val Acc: 72.66%
Val Score: 0.1883
New best model saved! Score: 0.1883

Epoch 5/5


              precision    recall  f1-score   support

           0       0.98      1.00      0.99      2627
           1       1.00      0.98      0.99      2692
           2       0.99      1.00      1.00      2763

    accuracy                           0.99      8082
   macro avg       0.99      0.99      0.99      8082
weighted avg       0.99      0.99      0.99      8082




CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.1000    0.0541    0.0702       351
     Class 1     0.7482    0.9238    0.8268      2821
     Class 2     0.1275    0.0292    0.0476       650

    accuracy                         0.6918      3822
   macro avg     0.3252    0.3357    0.3149      3822
weighted avg     0.5831    0.6918    0.6248      3822


Train Loss: 0.0131 | Train Acc: 99.20%
Val Loss: 2.0930 | Val Acc: 69.18%
Val Score: 0.1031


In [11]:
results = evaluate(best_model, test_loader, criterion, device)
print(f"Accuracy: {results['accuracy']:.2f}%")
print(f"F1 Score: {results['F1_score']:.4f}")
print(f"Quadratic Weighted Kappa: {results['Quadratic-weighted_Kappa']:.4f}")
print(f"Rk Correlation (Spearman): {results['Rk-correlation']:.4f}")
print(f"Specificity: {results['Specificity']:.4f}")
print(f"Score: {results['score']:.4f}")


CLASSIFICATION REPORT - Per-Class Performance:
              precision    recall  f1-score   support

     Class 0     0.4681    0.0761    0.1310       578
     Class 1     0.8523    0.9730    0.9086      4810
     Class 2     0.0000    0.0000    0.0000       321

    accuracy                         0.8275      5709
   macro avg     0.4401    0.3497    0.3465      5709
weighted avg     0.7655    0.8275    0.7788      5709


Accuracy: 82.75%
F1 Score: 0.8275
Quadratic Weighted Kappa: 0.0024
Rk Correlation (Spearman): 0.0993
Specificity: 0.6884
Score: 0.1729


In [12]:
from google.colab import drive
drive.mount('/content/drive')
#from google.colab import files
#files.download('model.pth')
import torch
torch.save(best_model.state_dict(), '/content/drive/MyDrive/best_model_mlp.pth')

MessageError: User cancelled dfs_ephemeral authorization